# Infer test filaments with a crop U-Net and write a submission CSV

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To use another model, change only `MODEL` in the settings cell and attach that model's training output:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

It loads the checkpoint written by `notebooks/train_unet_crops.ipynb`, predicts each competition test image with the same tiled inference that notebook uses for its holdout evaluation, and writes `/kaggle/working/submission.csv`. Test-time flip augmentation (`USE_TTA`) and a ResNet-18 + ResNet-34 ensemble (`USE_ENSEMBLE`) are optional and off by default, so the default run is the plain ResNet-34 (`MODEL` picks another encoder). It writes that file and stops. It does not submit it. There is no Kaggle submit call.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (the test images).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip, extracted as `Solar-Filament-Segmentation-main/`), a version that contains `filament.checkpoint.CROP_CHECKPOINTS`. [VERIFY] re-upload it after the merge.
3. The **output of the training notebook of `MODEL`**: by default `train-unet-r34-crops-80ep` (Add Input → Notebook Output). That assumes the training notebook was saved under the name `train-unet-r34-crops-80ep`. The default `WEIGHTS_PATH` is then `/kaggle/input/notebooks/ignacioaguirrecandia/train-unet-r34-crops-80ep/unet_r34_crops.pt`.
4. Only when `CROP_ENSEMBLE` is set: the **output of the training notebook of every other member**, for example `train-unet-r34-v2-agree` for `r34_v2_agree`. Each member loads from its entry's `weights_path` in `filament.checkpoint`, or from `ENSEMBLE_WEIGHTS_PATHS[name]`.
5. Only when `USE_ENSEMBLE = True`: the **output of the notebook `train-unet-r18-12ep-2026-10-31`** (the 512 px ResNet-18). The default `R18_WEIGHTS_PATH` comes from `filament.checkpoint.R18_12EP`.

**Settings**

- Accelerator: **GPU T4**, a single T4. Without a GPU the notebook **stops**, because efficiency is measured on a GPU. `ALLOW_CPU = True` runs it on the CPU anyway, only to check the plumbing: 180 images × 25 tiles of a ResNet-34 are very slow there, and the speed on CPU was not measured [VERIFY].
- Internet: **On**, for the `pip install` of `segmentation-models-pytorch==0.6.0`. The checkpoint holds the encoder, so no encoder weights are downloaded.


## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match:

`/kaggle/input/datasets/ignacioaguirrecandia/solar-code/Solar-Filament-Segmentation-main/src`

It is used in place and not copied.


In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT), the same version that trained the checkpoint. Kaggle already has PyTorch; this cell does not install it.


In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Settings and where the weights are

`MODEL` is a name of `filament.checkpoint.CROP_CHECKPOINTS`. `WEIGHTS_PATH` takes its default from that entry's `weights_path`, the single place that path is written. Set it to another `.pt` file to load that one, or to `None` to search every `*.pt` under `/kaggle/working` and `/kaggle/input` (skipping `competitions`).

If the default file does not exist, the notebook searches the same way and looks for a file or folder with that entry's file name (`unet_r34_crops.pt` for `r34`). One match is used. Without a match, a single `.pt` anywhere is used. Several `.pt` files and no single match stop the notebook and list every candidate.

Kaggle extracts an uploaded `.pt` because `torch.save` writes a zip. `unet_r34_crops.pt` can then become a directory that contains `unet_r34_crops/data.pkl`. That directory is not a checkpoint. The loader re-zips it to a real file under `/kaggle/working` and prints that path (the helper functions are the ones from `notebooks/infer_unet.ipynb`).

`MIN_AREA` 400 and `CLOSE_SIZE` 0 are the values that were best for the 512 px ResNet-18 on the holdout (`docs/RESULTS.md`). The `min_area` best for this ResNet-34 run is whatever `holdout_sweep_r34.csv` of the training notebook shows; change `MIN_AREA` only if you decide to follow it. It is one holdout, so a small difference is not a reason to change it.

`TILE_OVERLAP` (128) and `INFER_BATCH_SIZE` (8) are the training notebook's values, and the tile size is read from the checkpoint (`tile_size`, which is the crop size). `USE_AMP` matches the training notebook so the holdout score and the test predictions use the same arithmetic.

**Checkpoint guard.** The first U-Net submission scored LB 0.04 because it loaded the wrong weights. Before predicting, the notebook compares the checkpoint with the entry of `MODEL` (`filament.checkpoint`): the stored `epochs` and the encoder must match, and so must `tensor_sha256` once it is pinned (`r34` is pinned to the 80-epoch run of 2026-11-02; `r50` and `effb3` are not pinned yet, so the guard prints a `WARNING` that it did not check the weights). If the training run was stopped early by `MAX_TRAIN_HOURS`, its stored `epochs` is below the entry's and the guard stops here: set `EXPECTED_EPOCHS` to the stored number. `ALLOW_UNVERIFIED_WEIGHTS = True` turns a mismatch into a printed warning; leave it `False` for a real submission.

**Object filter.** `OBJECT_FILTER` drops a connected component when the mean, maximum or 90th percentile (`"stat"`) of its probability is below `"min"`. The kept components are not changed and stay disjoint. The default comes from `filament.components.ADOPTED_OBJECT_FILTER`, the setting the cross-fitted verdict of `notebooks/sweep_object_filter.ipynb` adopted: a component's mean probability at least 0.8 (`docs/RESULTS.md`). It was validated on flip-TTA maps at threshold 0.45, `close_size` 11, `min_area` 400 (`OBJECT_FILTER_VALIDATED_ON`). With other settings the notebook prints a `WARNING` and still runs. `OBJECT_FILTER = None` turns it off: the CSV is then the one the notebook wrote before the filter existed.

**Crop-model ensemble (off by default).** `CROP_ENSEMBLE = (("r34", 0.7), ("r34_v2_agree", 0.3))` averages the probability maps of several crop models, each after its own TTA, with weights normalized to sum to 1, before the threshold, the components and the object filter. `MODEL` has to be one of the members, and every member has to have a pinned `tensor_sha256` (the notebook stops otherwise, unless `ALLOW_UNVERIFIED_WEIGHTS`). Attach the training output of every member. Each member costs one more tiled pass per view. With `CROP_ENSEMBLE = None` the notebook writes exactly what it wrote before. Which weights to use comes from `notebooks/sweep_crop_ensemble.ipynb` on the holdout.

**TTA and ensemble (off by default).** `USE_TTA = True` averages four views of each frame (as is, flipped left-right, flipped top-bottom, both; each prediction flipped back). `USE_ENSEMBLE = True` also loads the 512 px ResNet-18 and averages its probability map with that of `MODEL`, weights `ENSEMBLE_WEIGHTS = (r18, MODEL)` normalized to sum to 1. The ResNet-18 predicts the frame area-averaged to 512 px and its map is resized back with bilinear interpolation; it is checked against `R18_12EP` (epochs, encoder, `tensor_sha256`) like `MODEL`. To submit the winner of `notebooks/sweep_tta_ensemble.ipynb`, set `USE_TTA`, `USE_ENSEMBLE`, `ENSEMBLE_WEIGHTS`, `THRESHOLD`, `CLOSE_SIZE` and `MIN_AREA` to the values of the best row of its `holdout_tta_ensemble.csv`. Each TTA view costs one more tiled pass: `USE_TTA` makes the prediction about four times as long, and `USE_ENSEMBLE` adds a cheap 512 px pass per view. The extra time on the 180 test images was not measured [VERIFY].

In [ ]:
import time
import zipfile

from filament.checkpoint import R18_12EP, crop_checkpoint
from filament.components import ADOPTED_OBJECT_FILTER

# ---- Settings ---------------------------------------------------------------
# Which model: a name of filament.checkpoint.CROP_CHECKPOINTS.
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# Default from the checkpoint config. Another .pt path loads that file; None
# searches /kaggle/working and /kaggle/input (skipping competitions).
WEIGHTS_PATH = EXPECTED.weights_path

# Crop-model ensemble. None (the default) is MODEL alone. Otherwise (name, weight)
# pairs of filament.checkpoint.CROP_CHECKPOINTS that include MODEL, for example
# ((MODEL, 0.7), ("r34_v2_agree", 0.3)). Weights are normalized to sum to 1. Each
# member's probability map, after its own TTA, is averaged before the threshold,
# the components and the object filter. Every member needs a pinned tensor_sha256.
# Members other than MODEL load from their entry's weights_path, or from
# ENSEMBLE_WEIGHTS_PATHS[name]. Not together with USE_ENSEMBLE (the ResNet-18).
CROP_ENSEMBLE = None
ENSEMBLE_WEIGHTS_PATHS = {}

# The checkpoint guard: stop unless the file matches. EXPECTED_TENSOR_SHA256 is
# None until the hash is pinned in filament.checkpoint.
EXPECTED_EPOCHS = EXPECTED.epochs
EXPECTED_TENSOR_SHA256 = EXPECTED.tensor_sha256
ALLOW_UNVERIFIED_WEIGHTS = False

# Without a GPU the notebook stops. True lets it run on the CPU (very slow),
# only to check the plumbing: efficiency is measured on a GPU.
ALLOW_CPU = False

# Postprocess. min_area is for a 2048 x 2048 frame; close_size 0 is no closing.
MIN_AREA = 400
CLOSE_SIZE = 0

# Object filter (roadmap B1): drop a component whose probability statistic is below
# "min", for example {"stat": "mean", "min": 0.8}. The default is the setting the
# B1 sweep adopted by its cross-fitted verdict (docs/RESULTS.md). None turns it
# off and gives exactly the CSV this notebook wrote before the filter existed.
OBJECT_FILTER = ADOPTED_OBJECT_FILTER

# Tiled inference, as in the training notebook's holdout evaluation.
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
THRESHOLD = 0.5
USE_AMP = True

# Test-time augmentation and ensemble, both off. Set them, and THRESHOLD,
# CLOSE_SIZE and MIN_AREA above, to the best row of holdout_tta_ensemble.csv.
USE_TTA = False
USE_ENSEMBLE = False
# (ResNet-18 weight, MODEL weight); normalized to sum to 1.
ENSEMBLE_WEIGHTS = (0.5, 0.5)
R18_WEIGHTS_PATH = R18_12EP.weights_path
R18_SIDE = 512

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
PREFERRED_NAME = Path(EXPECTED.weights_path).name


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


def _resolve_weights(explicit):
    """Return the checkpoint file to load.

    ``explicit`` that exists is used (an extracted archive is re-zipped). The
    default path missing, or ``None``, searches ``/kaggle/working`` and
    ``/kaggle/input``: a file or folder named ``PREFERRED_NAME`` first, then
    a lone ``.pt``. A path that was set by hand and is missing is an error.
    """
    if explicit is not None:
        path = Path(explicit)
        if path.exists():
            return _materialize_checkpoint(path)
        if str(explicit) != EXPECTED.weights_path:
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist."
            )
        print(f"{path} does not exist. Searching for {PREFERRED_NAME}.")
    candidates = _weight_candidates()
    named = [path for path in candidates if path.name == PREFERRED_NAME]
    if len(named) == 1:
        return _materialize_checkpoint(named[0])
    if not named and len(candidates) == 1:
        print(f"No {PREFERRED_NAME}; using the only .pt found: {candidates[0]}")
        return _materialize_checkpoint(candidates[0])
    listed = "\n".join(f"  {path}" for path in candidates) or "  (none)"
    detail = (
        f"Several files named {PREFERRED_NAME} were found."
        if named
        else f"No {PREFERRED_NAME} and not exactly one .pt were found."
    )
    raise RuntimeError(
        f"{detail} Attach the output of the notebook "
        f"{Path(EXPECTED.weights_path).parent.name}, or "
        f"set WEIGHTS_PATH to the file to load.\n{listed}"
    )

## 4. Load the checkpoint and check it

The fingerprint printed here is the resolved path, the file size, the MD5 of the file bytes, the stored fields, and `tensor_sha256` (SHA-256 of the `state_dict` tensor bytes, from `filament.checkpoint.tensor_sha256`, the same function the training notebook uses). The MD5 of a re-zipped file is not comparable to the original file's MD5; compare `tensor_sha256`.

The guard then runs, and the network is rebuilt from the checkpoint with `encoder_weights=None`. Seed 0 is set. The device is CUDA when there is one.

With `USE_ENSEMBLE = True` the ResNet-18 checkpoint is opened with the same helpers, checked against `R18_12EP`, and rebuilt with `encoder_weights=None`. Its stored `side` has to be 512.


In [ ]:
import hashlib
import random

import numpy as np
import torch

from filament import crop_training as ct
from filament.checkpoint import checkpoint_fingerprint, verify_and_report
from filament.crop_ensemble import members_to_load, submission_model_weights
from filament.ensemble import EvalConfig, ensemble_predictor

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda" and not ALLOW_CPU:
    raise RuntimeError(
        "No GPU found. Efficiency is measured on a GPU, so this notebook "
        "stops. On Kaggle set Settings > Accelerator > GPU T4. "
        "ALLOW_CPU = True in the settings cell runs it on the CPU anyway, "
        "only to check the plumbing."
    )
if device.type == "cuda":
    print(f"device={torch.cuda.get_device_name(0)}")
else:
    print(
        "WARNING: device=cpu (ALLOW_CPU). Predicting 180 images tile by tile will be slow."
    )
use_amp = USE_AMP and device.type == "cuda"

# Which models, and with which weights. Checked before any weights load: a bad
# CROP_ENSEMBLE, an unpinned member, or both ensembles at once stop here.
MODEL_WEIGHTS = submission_model_weights(
    MODEL,
    CROP_ENSEMBLE,
    use_r18_ensemble=USE_ENSEMBLE,
    r18_weights=ENSEMBLE_WEIGHTS,
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
)

WEIGHTS_FILE = _resolve_weights(WEIGHTS_PATH)
print(f"loaded weights={WEIGHTS_FILE}")
checkpoint = ct.load_checkpoint(WEIGHTS_FILE)
fingerprint = checkpoint_fingerprint(checkpoint)

data = WEIGHTS_FILE.read_bytes()
print(f"bytes={len(data)} file_md5={hashlib.md5(data).hexdigest()}")
del data
print(
    "File MD5 of a re-zip is not comparable to the original file MD5. "
    "Compare tensor_sha256."
)
for key in (
    "epochs",
    "epochs_requested",
    "status",
    "encoder_name",
    "crop_size",
    "training_resolution",
    "tile_size",
    "tile_overlap",
    "smp_version",
):
    print(f"  {key}={checkpoint.get(key, 'not stored')}")
print(f"  tensor_sha256={fingerprint['tensor_sha256']}")

# Raises CheckpointMismatchError on the wrong weights unless
# ALLOW_UNVERIFIED_WEIGHTS is True. Prints a WARNING while the hash is unpinned.
verify_and_report(
    fingerprint,
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
    expected_epochs=EXPECTED_EPOCHS,
    expected_tensor_sha256=EXPECTED_TENSOR_SHA256,
)

TILE_SIZE = int(checkpoint.get("tile_size") or checkpoint["crop_size"])
saved_overlap = checkpoint.get("tile_overlap")
if saved_overlap is not None and int(saved_overlap) != TILE_OVERLAP:
    print(
        f"WARNING: the training notebook evaluated the holdout with overlap "
        f"{saved_overlap}; TILE_OVERLAP here is {TILE_OVERLAP}."
    )
model = ct.model_from_checkpoint(checkpoint, device)
del checkpoint

PREDICTORS = {
    MODEL: ct.tiled_predictor(
        model,
        device,
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
        batch_size=INFER_BATCH_SIZE,
        use_amp=use_amp,
    )
}
for name in members_to_load(MODEL_WEIGHTS, MODEL):
    member = crop_checkpoint(name)
    member_path = Path(ENSEMBLE_WEIGHTS_PATHS.get(name, member.weights_path))
    if not member_path.exists():
        raise FileNotFoundError(
            f"Ensemble member {name!r}: {member_path} does not exist. Attach the "
            f"output of the notebook {Path(member.weights_path).parent.name}, or "
            "set ENSEMBLE_WEIGHTS_PATHS."
        )
    member_file = _materialize_checkpoint(member_path)
    print(f"{name} weights={member_file}")
    member_checkpoint = ct.load_checkpoint(member_file)
    member_fingerprint = checkpoint_fingerprint(member_checkpoint)
    print(f"  tensor_sha256={member_fingerprint['tensor_sha256']}")
    verify_and_report(
        member_fingerprint,
        allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
        expected=member,
    )
    member_tile = int(
        member_checkpoint.get("tile_size") or member_checkpoint["crop_size"]
    )
    PREDICTORS[name] = ct.tiled_predictor(
        ct.model_from_checkpoint(member_checkpoint, device),
        device,
        tile_size=member_tile,
        overlap=TILE_OVERLAP,
        batch_size=INFER_BATCH_SIZE,
        use_amp=use_amp,
    )
    del member_checkpoint
if USE_ENSEMBLE:
    r18_path = Path(R18_WEIGHTS_PATH)
    if not r18_path.exists():
        raise FileNotFoundError(
            f"USE_ENSEMBLE is True but {r18_path} does not exist. Attach the "
            "output of the notebook train-unet-r18-12ep-2026-10-31, or set "
            "R18_WEIGHTS_PATH."
        )
    r18_file = _materialize_checkpoint(r18_path)
    print(f"r18 weights={r18_file}")
    checkpoint18 = ct.load_checkpoint(r18_file)
    fingerprint18 = checkpoint_fingerprint(checkpoint18)
    print(f"  tensor_sha256={fingerprint18['tensor_sha256']}")
    verify_and_report(
        fingerprint18,
        allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
        expected=R18_12EP,
    )
    PREDICTORS["r18"] = ct.resized_predictor(
        ct.whole_frame_model_from_checkpoint(checkpoint18, device, side=R18_SIDE),
        device,
        side=R18_SIDE,
    )
    del checkpoint18
CONFIG = EvalConfig("submission", MODEL_WEIGHTS, tta=USE_TTA)
predict = ensemble_predictor(PREDICTORS, CONFIG)
print(f"tta={USE_TTA} ensemble={USE_ENSEMBLE} model weights={MODEL_WEIGHTS}")
print(f"tile_size={TILE_SIZE} overlap={TILE_OVERLAP} batch={INFER_BATCH_SIZE}")
print(f"seed={SEED} amp={use_amp}")

## 5. Predict the test images and write the CSV

Test images, JPEG files, and not the train set:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images`

Each image is read at 2048 × 2048. `filament.crop_training.predict_probability` runs the overlapped tiles and averages the probabilities (the same call the training notebook makes for the holdout; with `USE_TTA` or `USE_ENSEMBLE` it is called through `filament.ensemble.ensemble_predictor`), the threshold is 0.5, and instances are `filament.instances.labels_from_semantic` at full resolution with `MIN_AREA` and `CLOSE_SIZE`. Then `filament.components.filter_components` drops the components below `OBJECT_FILTER`, if it is set, before any of them is encoded.

`filament.submission.write_submission` writes `/kaggle/working/submission.csv` with columns `filament_id` and `segmentation_rle`. `validate_submission` (`filament.validate`) then checks that file. This cell does not submit it.


In [ ]:
from filament.baseline import iter_filament_counts
from filament.components import (
    filter_components,
    object_cut_from_config,
    object_filter_mismatches,
)
from filament.crop_data import read_gray
from filament.instances import labels_from_semantic
from filament.submission import filament_id, write_submission
from filament.validate import validate_submission

TEST_IMAGES = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/test/test_images"
)
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
FULL_SIDE = 2048

OBJECT_CUT = object_cut_from_config(OBJECT_FILTER)
if OBJECT_CUT is not None:
    for mismatch in object_filter_mismatches(
        threshold=THRESHOLD,
        close_size=CLOSE_SIZE,
        min_area=MIN_AREA,
        tta=USE_TTA,
        models=MODEL_WEIGHTS,
    ):
        print(f"WARNING: object filter {OBJECT_CUT} is untested with {mismatch}.")


def _test_paths() -> list[Path]:
    if not TEST_IMAGES.is_dir():
        raise FileNotFoundError(
            f"Test images were not found at {TEST_IMAGES}. Attach the competition."
        )
    paths = sorted(
        path
        for path in TEST_IMAGES.iterdir()
        if path.is_file() and path.suffix.lower() == ".jpeg"
    )
    if not paths:
        raise FileNotFoundError(f"No .jpeg files in {TEST_IMAGES}")
    return paths


inference_seconds = 0.0


def _iter_rows(paths: list[Path]):
    global inference_seconds
    total = len(paths)
    for index, path in enumerate(paths, start=1):
        gray = read_gray(path, (FULL_SIDE, FULL_SIDE))
        begin = time.monotonic()
        probability = predict(gray)
        inference_seconds += time.monotonic() - begin
        semantic = ct.semantic_from_probability(probability, THRESHOLD)
        labels = labels_from_semantic(
            semantic, gray, min_area=MIN_AREA, close_size=CLOSE_SIZE
        )
        labels = filter_components(labels, probability, OBJECT_CUT)
        count = 0
        for number, counts in iter_filament_counts(labels):
            count += 1
            yield filament_id(path.stem, number), counts
        del gray, probability, semantic, labels
        if index == 1 or index % 20 == 0 or index == total:
            print(f"{index}/{total} {path.name} filaments={count}", flush=True)


def _write_predictions() -> None:
    started = time.monotonic()
    paths = _test_paths()
    print(f"test images={len(paths)} directory={TEST_IMAGES}")
    print(
        f"tile={TILE_SIZE} overlap={TILE_OVERLAP} threshold={THRESHOLD} "
        f"min_area={MIN_AREA} close_size={CLOSE_SIZE} "
        f"tta={USE_TTA} ensemble={USE_ENSEMBLE} models={MODEL_WEIGHTS} "
        f"object_filter={OBJECT_CUT}"
    )
    print("train images are not read")
    SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    def _counted():
        nonlocal written
        for row in _iter_rows(paths):
            written += 1
            yield row

    write_submission(SUBMISSION_PATH, _counted())
    validate_submission(SUBMISSION_PATH)
    print(f"wrote {written} rows to {SUBMISSION_PATH}")
    print("columns=filament_id,segmentation_rle")
    print(
        f"tiled inference {inference_seconds:.0f} s, "
        f"total {time.monotonic() - started:.0f} s"
    )
    print("This notebook wrote the CSV. It did not submit it.")


_write_predictions()